In [38]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [29]:
df=pd.read_csv('/content/sample_data/wine.csv')

In [ ]:
df.head()

In [ ]:
df.dtypes
df.info()

In [ ]:
df.describe().T

In [ ]:
#Correlation
cor=df.corr()
cor.loc[1,1] = np.tril(cor, k=-1)
cor=cor.stack()
cor[(cor > 0.55) | (cor <= -0.55)]


In [ ]:
cor=df.corr()
cor

In [ ]:
Target_Inb = df['quality'].value_counts(normalize=True)
Target_Inb

In [ ]:
df['quality'] = df['quality'].replace(8,7)
df['quality'] = df['quality'].replace(3,5)
df['quality'] = df['quality'].replace(4,5)
df['quality'].value_counts()

In [ ]:
#Splitting the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('quality', axis=1), df['quality'], 
                                                    test_size=.3, random_state=10)
X_train.shape, X_test.shape

In [ ]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
model1_entropy = DecisionTreeClassifier(criterion='entropy')
model1_entropy.fit(X_train, y_train)
model1_entropy.score(X_train, y_train)
model1_entropy.score(X_test, y_test)

In [ ]:
clf_pruned=DecisionTreeClassifier(criterion='entropy', random_state=100, max_depth=5, min_samples_leaf=5)
clf_pruned.fit(X_train, y_train)
pred_pruned_train = clf_pruned.predict(X_train)
pred_pruned = clf_pruned.predict(X_test)
print(metrics.accuracy_score(y_train, pred_pruned_train))
print(metrics.accuracy_score(y_test, pred_pruned))

In [ ]:
acc_DT = metrics.accuracy_score(y_test, pred_pruned)
resultsDf = pd.DataFrame({'Method' : ['Decision Tree'], 'accuracy' : acc_DT})
resultsDf = resultsDf[['Method', 'accuracy']]
resultsDf


In [69]:
#Apply Random Forest model
from sklearn.ensemble import RandomForestClassifier
rfcl = RandomForestClassifier(n_estimators = 50)
rfcl = rfcl.fit(X_train, y_train)

In [71]:
pred_RF = rfcl.predict(X_test)
acc_RF = metrics.accuracy_score(y_test, pred_RF)

In [ ]:
tempResultsDf = pd.DataFrame({'Method' : ['Random Forest'], 'accuracy' : acc_RF})
resultsDf = pd.concat([resultsDf, tempResultsDf])
resultsDf = resultsDf[['Method', 'accuracy']]
resultsDf

In [74]:
#Apply Adaboost
from sklearn.ensemble import AdaBoostClassifier
abcl = AdaBoostClassifier(n_estimators = 100, learning_rate=0.1, random_state=22)
abcl = abcl.fit(X_train, y_train)

In [75]:
pred_AB = abcl.predict(X_test)
acc_AB = metrics.accuracy_score(y_test, pred_AB)

In [ ]:
tempResultsDf = pd.DataFrame({'Method' : ['AdaBoost'], 'accuracy' : acc_AB})
resultsDf = pd.concat([resultsDf, tempResultsDf])
resultsDf = resultsDf[['Method', 'accuracy']]
resultsDf

In [77]:
#Apply Bagging Classifier
from sklearn.ensemble import BaggingClassifier
bgcl = BaggingClassifier(n_estimators=50, bootstrap=True, oob_score=True, random_state=22)
bgcl = bgcl.fit(X_train, y_train)

In [78]:
pred_BG = bgcl.predict(X_test)
acc_BG = metrics.accuracy_score(y_test, pred_BG)

In [ ]:
tempResultsDf = pd.DataFrame({'Method' : ['Boosting'], 'accuracy' : acc_BG})
resultsDf = pd.concat([resultsDf, tempResultsDf])
resultsDf = resultsDf[['Method', 'accuracy']]
resultsDf

In [80]:
#Apply Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
gbcl = GradientBoostingClassifier(n_estimators=50, learning_rate = 0.1, random_state=22)
gbcl = gbcl.fit(X_train, y_train)

In [82]:
pred_GB = gbcl.predict(X_test)
acc_GB = metrics.accuracy_score(y_test, pred_BG)

In [ ]:
tempResultsDf = pd.DataFrame({'Method' : ['Gradient Boosting'], 'accuracy' : acc_GB})
resultsDf = pd.concat([resultsDf, tempResultsDf])
resultsDf = resultsDf[['Method', 'accuracy']]
resultsDf

In [ ]:
sns.countplot(df['quality'])

In [ ]:
sns.distplot(df['fixed acidity'])

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(df.corr(),
            annot=True,
            linewidths=.5,
            center=0,
            cbar=False,
            cmap='RdYlGn')


In [ ]:
dot_data = StringIO()
export_graphviz(clf_pruned, out_file=dot_data,
                filled=True, rounded=True,
                special_characters=True,feature_names=feature_cols, class_names=['0', '1', '2'] )
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
graph.write_png('wines_pruned.png')
Image(graph.create_png())

In [85]:
#feature importance
feat_importance = clf_pruned.tree_.compute_feature_importances(normalize=False)
feat_imp_dict = dict(zip(feature_cols, clf_pruned.feature_importances_))
feat_imp = pd.DataFrame.from_dict(feat_imp_dict, orient='index')
feat_imp.sort_values(by=0, ascending=False)

,0
alcohol,0.392647
total sulfur dioxide,0.186004
sulphates,0.168333
volatile acidity,0.122200
chlorides,0.049732
pH,0.025498
density,0.020249
free sulfur dioxide,0.018078
citric acid,0.010286
residual sugar,0.006975
